# フラッピーバード風ゲーム（p5.js ゲーム）

重力で落ちる鳥をジャンプさせて、土管のすき間をくぐり抜けるゲームです。

## ルール
- 土管のすき間を通り抜けるごとに **1 点**
- 土管・地面・天井にぶつかるとゲームオーバー

## 操作
- **スペースキー または クリック**: ジャンプ（先にゲーム画面をクリックしてください）
- ゲームオーバー後は **スペースキー または クリック** でもう一度

## このノートブックの使い方

- コードセルを上から順番に **Shift + Enter** で実行し、最後の `%show` セルを実行するとゲーム画面が表示されます。
- キーボードで操作するゲームは、**最初にゲーム画面をクリック** してから操作してください（クリックでキー入力が画面に届くようになります）。
- コードを書き換えたら、そのセルを実行し直してから `%show` をもう一度実行すると、新しいゲームになります。
- 動かなくなったら、メニューの **Kernel → Restart Kernel and Clear Outputs of All Cells...** で最初からやり直せます。

p5.js の基本は `p5-tutorial.ipynb` で学べます。

## 1. ゲームの状態と鳥

鳥は「y 座標」と「縦方向の速度」を持ちます。毎フレーム速度に重力を足し、速度を位置に足すと自然な落下になります。ジャンプは速度を上向きの値に置き換えるだけです。

In [ ]:
const GRAVITY = 0.5;       // 重力（毎フレーム速度に足す）
const JUMP = -8;           // ジャンプしたときの速度（上向きはマイナス）
const BIRD_X = 100;        // 鳥の x 座標（固定。土管の方が動く）
const BIRD_R = 14;         // 鳥の半径

let birdY = 200;
let birdVY = 0;
let pipes = [];
let score = 0;
let state = "ready";       // "ready", "play", "gameover"

## 2. 土管のクラス

土管は「x 座標・すき間の上端・すき間の高さ」を持ち、左に流れていきます。すき間の位置はランダムです。
`passed` は「この土管を通過してスコアを加算したか」を覚えるためのフラグです。

In [ ]:
class Pipe {
  constructor() {
    this.x = width;                                  // 右端から登場
    this.w = 50;                                     // 土管の幅
    this.gap = 120;                                  // すき間の高さ
    this.top = random(50, height - 50 - this.gap);   // すき間の上端
    this.speed = 3;
    this.passed = false;
  }

  update() {
    this.x -= this.speed;
  }

  show() {
    fill(80, 200, 80);
    stroke(40, 120, 40);
    strokeWeight(2);
    rect(this.x, 0, this.w, this.top);                                        // 上の土管
    rect(this.x, this.top + this.gap, this.w, height - this.top - this.gap);  // 下の土管
  }

  // 鳥（中心 BIRD_X, birdY, 半径 r）が土管に当たっているか
  hits(y, r) {
    const inX = BIRD_X + r > this.x && BIRD_X - r < this.x + this.w;
    const inGap = y - r > this.top && y + r < this.top + this.gap;
    return inX && !inGap;
  }

  isOffscreen() {
    return this.x + this.w < 0;
  }
}

## 3. setup と draw

In [ ]:
function setup() {
  createCanvas(400, 400);
  textFont("sans-serif");
}

function draw() {
  background(120, 200, 255);

  if (state === "play") {
    // 鳥の物理
    birdVY += GRAVITY;
    birdY += birdVY;

    // 90 フレームごとに土管を追加
    if (frameCount % 90 === 0) {
      pipes.push(new Pipe());
    }

    for (let i = pipes.length - 1; i >= 0; i--) {
      const p = pipes[i];
      p.update();
      if (p.hits(birdY, BIRD_R)) {
        state = "gameover";
      }
      if (!p.passed && p.x + p.w < BIRD_X) {   // 鳥が土管を通り過ぎた
        p.passed = true;
        score++;
      }
      if (p.isOffscreen()) {
        pipes.splice(i, 1);
      }
    }

    // 地面・天井
    if (birdY + BIRD_R > height || birdY - BIRD_R < 0) {
      state = "gameover";
    }
  }

  for (const p of pipes) p.show();
  drawBird();
  drawHUD();
}

function drawBird() {
  noStroke();
  fill(255, 220, 0);
  circle(BIRD_X, birdY, BIRD_R * 2);
  fill(255);
  circle(BIRD_X + 5, birdY - 4, 8);          // 目
  fill(0);
  circle(BIRD_X + 6, birdY - 4, 4);
  fill(255, 140, 0);
  triangle(BIRD_X + 12, birdY, BIRD_X + 22, birdY + 3, BIRD_X + 12, birdY + 6);   // くちばし
}

function drawHUD() {
  fill(255);
  stroke(0);
  strokeWeight(3);
  textSize(28);
  textAlign(CENTER, TOP);
  text(score, width / 2, 10);

  noStroke();
  textAlign(CENTER, CENTER);
  if (state === "ready") {
    fill(0);
    textSize(18);
    text("スペースキーかクリックでスタート", width / 2, height / 2 + 60);
  } else if (state === "gameover") {
    fill(0, 170);
    rect(0, 0, width, height);
    fill(255);
    textSize(36);
    text("ゲームオーバー", width / 2, height / 2 - 20);
    textSize(16);
    text("スコア: " + score + "　スペースキーかクリックでもう一度", width / 2, height / 2 + 25);
  }
}

## 4. 入力とリセット

クリックとスペースキーで同じ処理をするので、共通の関数 `flap()` にまとめます。

In [ ]:
function flap() {
  if (state === "ready") {
    state = "play";
    birdVY = JUMP;
  } else if (state === "play") {
    birdVY = JUMP;
  } else if (state === "gameover") {
    resetGame();
  }
}

function mousePressed() {
  flap();
}

function keyPressed() {
  if (key === " ") {
    flap();
    return false;   // スペースキーでページがスクロールするのを防ぐ
  }
}

function resetGame() {
  birdY = 200;
  birdVY = 0;
  pipes = [];
  score = 0;
  state = "ready";
}

In [ ]:
%show 100% 410px

## 改造のヒント

- `GRAVITY` や `JUMP`、土管の `gap` を変えて難易度を調整してみましょう
- スコアが上がるほど土管の `speed` を速くしてみましょう
- 鳥の速度に合わせて `rotate()` で傾けると、それらしく見えます
- 背景に雲や地面を描いてスクロールさせてみましょう